# ============================================================
# MDE-CS-03_PRINCIPLES OF STATISTICAL MODELLING
# FULL PROJECT NOTEBOOK
# Name: Subhankar Biswas
# Matriculation Number: 30008967
# Dataset: Wine Quality Dataset (winequality-red.csv)
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import (
    skew,
    kurtosis,
    norm,
    gamma,
    multivariate_normal
)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 2. LOAD DATASET
# ============================================================

# Download from:
# https://archive.ics.uci.edu/dataset/186/wine+quality


In [ ]:
file_path = "raw-data/winequality-red.csv"

df = pd.read_csv(file_path, sep=";")

print("===================================================")
print("DATASET INFORMATION")
print("===================================================")

print(df.head())
print("\nDataset Shape:", df.shape)

# ============================================================
# 3. MATHEMATICAL FORMALISM
# ============================================================

In [ ]:
print("""
===================================================
MATHEMATICAL FORMALISM
===================================================

Universe:
Ω = {ω1, ω2, ..., ωN}

Random Vector:
X = (X1, X2, ..., Xd)

Features include:
- fixed acidity
- volatile acidity
- citric acid
- alcohol
- density
- etc.

Value Space:
S = S1 × S2 × ... × Sd
""")

# ============================================================
# 4. DATA CLEANING
# ============================================================

In [ ]:
print("===================================================")
print("MISSING VALUES")
print("===================================================")

print(df.isnull().sum())

print("""
No missing values detected.
Dataset is already clean.
""")

# ============================================================
# 5. RAW DATA VISUALIZATION
# ============================================================

In [ ]:
print("===================================================")
print("RAW DATA DESCRIPTION")
print("===================================================")

print(df.describe())

# ============================================================
# 6. DISTRIBUTION + MOMENTS
# ============================================================

In [ ]:
print("""
===================================================
Distribution and Moments
===================================================
""")

# ============================================================
# 7. SELECT FEATURE
# ============================================================

In [ ]:
X = df["alcohol"]

print("Selected Feature: Alcohol")

# ============================================================
# 8. DISCRETIZATION
# ============================================================

In [ ]:
bin_width = 0.5

bins = np.arange(X.min(), X.max() + bin_width, bin_width)

print("Bins:")
print(bins)

# ============================================================
# 9. HISTOGRAM
# ============================================================

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    X,
    bins=bins,
    edgecolor='black'
)

plt.title("Histogram of Alcohol")
plt.xlabel("Alcohol")
plt.ylabel("Frequency")

plt.grid(True)

plt.show()

# ============================================================
# 10. PMF
# ============================================================

In [ ]:
counts, edges = np.histogram(X, bins=bins)

pmf = counts / len(X)

plt.figure(figsize=(8,5))

plt.bar(
    edges[:-1],
    pmf,
    width=(edges[1]-edges[0]),
    align='edge'
)

plt.title("PMF of Alcohol")
plt.xlabel("Alcohol Bins")
plt.ylabel("Probability")

plt.grid(True)

plt.show()

# ============================================================
# 11. MOMENTS
# ============================================================

In [ ]:
mean_val = np.mean(X)
std_val = np.std(X)
skew_val = skew(X)
kurt_val = kurtosis(X)

print("===================================================")
print("MOMENTS")
print("===================================================")

print("Mean:", mean_val)
print("Standard Deviation:", std_val)
print("Skewness:", skew_val)
print("Kurtosis:", kurt_val)

# ============================================================
# 12. NORMAL DISTRIBUTION FIT
# ============================================================

In [ ]:
pdf_normal = norm.pdf(edges[:-1], mean_val, std_val)

rmse_normal = np.sqrt(
    np.mean((pmf - pdf_normal)**2)
)

print("===================================================")
print("NORMAL DISTRIBUTION")
print("===================================================")

print("RMSE (Normal):", rmse_normal)

# ============================================================
# 13. GAMMA DISTRIBUTION FIT
# ============================================================

In [ ]:
shape, loc, scale = gamma.fit(X)

pdf_gamma = gamma.pdf(
    edges[:-1],
    shape,
    loc=loc,
    scale=scale
)

rmse_gamma = np.sqrt(
    np.mean((pmf - pdf_gamma)**2)
)

print("===================================================")
print("GAMMA DISTRIBUTION")
print("===================================================")

print("Shape:", shape)
print("Scale:", scale)
print("RMSE (Gamma):", rmse_gamma)

# ============================================================
# 14. COMPARE DISTRIBUTIONS
# ============================================================

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    edges[:-1],
    pmf,
    width=(edges[1]-edges[0]),
    alpha=0.5,
    label='Empirical PMF'
)

plt.plot(
    edges[:-1],
    pdf_normal,
    label='Normal Fit'
)

plt.plot(
    edges[:-1],
    pdf_gamma,
    label='Gamma Fit'
)

plt.title("Distribution Comparison")
plt.xlabel("Alcohol")
plt.ylabel("Probability")

plt.legend()
plt.grid(True)

plt.show()

# ============================================================
# 15. INTERPRETATION
# ============================================================

In [ ]:
print("""
===================================================
INTERPRETATION
===================================================

1. Alcohol distribution is slightly right-skewed.

2. Gamma distribution fits the data better than
   the normal distribution.

3. Real-world data is not perfectly Gaussian.
""")

# ============================================================
# 16. CORRELATION + JOINT PMF
# ============================================================

In [ ]:
print("""
===================================================
Correlation and Joint PMF
===================================================
""")

# ============================================================
# 17. SELECT TWO FEATURES
# ============================================================

In [ ]:
X1 = df["alcohol"]
X2 = df["density"]

# ============================================================
# 18. HISTOGRAMS
# ============================================================

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(X1, bins=10, edgecolor='black')

plt.title("Alcohol Distribution")
plt.xlabel("Alcohol")
plt.ylabel("Frequency")

plt.grid(True)

plt.show()


plt.figure(figsize=(8,5))

plt.hist(X2, bins=10, edgecolor='black')

plt.title("Density Distribution")
plt.xlabel("Density")
plt.ylabel("Frequency")

plt.grid(True)

plt.show()

# ============================================================
# 19. SCATTER PLOT
# ============================================================

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    X1,
    X2,
    alpha=0.6
)

plt.title("Scatter Plot: Alcohol vs Density")
plt.xlabel("Alcohol")
plt.ylabel("Density")

plt.grid(True)

plt.show()

# ============================================================
# 20. PEARSON CORRELATION
# ============================================================

In [ ]:
pearson_corr = X1.corr(X2)

print("===================================================")
print("PEARSON CORRELATION")
print("===================================================")

print("Correlation:", pearson_corr)

if pearson_corr < 0:
    print("Variables are anticorrelated.")
elif pearson_corr > 0:
    print("Variables are positively correlated.")
else:
    print("Variables are uncorrelated.")

# ============================================================
# 21. JOINT PMF
# ============================================================

In [ ]:
joint_counts, xedges, yedges = np.histogram2d(
    X1,
    X2,
    bins=20
)

joint_pmf = joint_counts / np.sum(joint_counts)

print("Joint PMF Shape:", joint_pmf.shape)

# ============================================================
# 22. 3D JOINT PMF PLOT
# ============================================================

In [ ]:
fig = plt.figure(figsize=(10,10))

ax = fig.add_subplot(111, projection='3d')

xpos, ypos = np.meshgrid(
    xedges[:-1],
    yedges[:-1],
    indexing="ij"
)

xpos = xpos.ravel()
ypos = ypos.ravel()

zpos = np.zeros_like(xpos)

dx = 0.3 * np.ones_like(zpos)
dy = 0.0005 * np.ones_like(zpos)

dz = joint_pmf.ravel()

ax.bar3d(
    xpos,
    ypos,
    zpos,
    dx,
    dy,
    dz,
    shade=True
)

ax.set_title("3D Joint PMF")
ax.set_xlabel("Alcohol")
ax.set_ylabel("Density")
ax.set_zlabel("Probability")

plt.show()

# ============================================================
# 23. STANDARD DEVIATIONS
# ============================================================

In [ ]:
sigma1 = np.std(X1)
sigma2 = np.std(X2)

print("===================================================")
print("STANDARD DEVIATIONS")
print("===================================================")

print("Sigma Alcohol:", sigma1)
print("Sigma Density:", sigma2)

# ============================================================
# 24. COVARIANCE MATRIX
# ============================================================

In [ ]:
cov_matrix = np.cov(X1, X2)

print("===================================================")
print("COVARIANCE MATRIX")
print("===================================================")

print(cov_matrix)

# ============================================================
# 25. BIVARIATE GAUSSIAN
# ============================================================

In [ ]:
mu = [np.mean(X1), np.mean(X2)]

rv = multivariate_normal(
    mean=mu,
    cov=cov_matrix
)

# ============================================================
# 26. CREATE GRID
# ============================================================

In [ ]:
x = np.linspace(X1.min(), X1.max(), 100)
y = np.linspace(X2.min(), X2.max(), 100)

Xg, Yg = np.meshgrid(x, y)

pos = np.dstack((Xg, Yg))

Z = rv.pdf(pos)

# ============================================================
# 27. PLOT BIVARIATE GAUSSIAN
# ============================================================

In [ ]:
fig = plt.figure(figsize=(10,10))

ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(
    Xg,
    Yg,
    Z,
    cmap='viridis'
)

ax.set_title("Bivariate Gaussian")
ax.set_xlabel("Alcohol")
ax.set_ylabel("Density")
ax.set_zlabel("Probability Density")

plt.show()

# ============================================================
# 28. CORRELATION HEATMAP
# ============================================================

In [ ]:
plt.figure(figsize=(10,8))

corr_matrix = df.corr()

plt.imshow(corr_matrix)

plt.colorbar()

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns
)

plt.title("Correlation Heatmap")

plt.show()

# ============================================================
# 29. PCA ANALYSIS
# ============================================================

In [ ]:
X_features = df.drop("quality", axis=1)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_features)

pca = PCA()

X_pca = pca.fit_transform(X_scaled)

print("===================================================")
print("PCA")
print("===================================================")

print("Explained Variance Ratios:")
print(pca.explained_variance_ratio_)

# ============================================================
# 30. PCA VARIANCE PLOT
# ============================================================

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    np.cumsum(pca.explained_variance_ratio_),
    marker='o'
)

plt.title("Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Variance Explained")

plt.grid(True)

plt.show()

# ============================================================
# 31. PCA SCATTER PLOT
# ============================================================

In [ ]:
plt.figure(figsize=(8,8))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    alpha=0.6
)

plt.title("PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")

plt.grid(True)

plt.show()

# ============================================================
# 32. LINEAR REGRESSION
# ============================================================

In [ ]:
X_reg = df.drop("quality", axis=1)
y_reg = df["quality"]

model = LinearRegression()

model.fit(X_reg, y_reg)

print("===================================================")
print("LINEAR REGRESSION")
print("===================================================")

print("R^2 Score:", model.score(X_reg, y_reg))

print("\nCoefficients:")

for feature, coef in zip(X_reg.columns, model.coef_):
    print(feature, ":", coef)

# ============================================================
# 33. OUTLIER DETECTION
# ============================================================

In [ ]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)

IQR = Q3 - Q1

outliers = (
    (
        df < (Q1 - 1.5 * IQR)
    ) |
    (
        df > (Q3 + 1.5 * IQR)
    )
).sum()

print("===================================================")
print("OUTLIERS")
print("===================================================")

print(outliers)

# ============================================================
# 34. FINAL CONCLUSION
# ============================================================

In [ ]:
print("""
===================================================
FINAL CONCLUSION
===================================================

1. The Wine Quality dataset exhibits non-Gaussian
   statistical structure.

2. Alcohol distribution is positively skewed.

3. Gamma distribution fits better than Gaussian.

4. Alcohol and density are anticorrelated.

5. PCA reveals latent low-dimensional structure.

6. Linear regression partially predicts wine quality.

7. Real-world datasets contain noise, correlations,
   and complex dependencies.
""")